In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np

# Load data
df = pd.read_csv('/Users/gautam/Desktop/MinneMUDAC2025/updated_mathclength_sorted_Training.csv', low_memory=False)
df = df.dropna(subset=['Completion Date', 'Match Support Contact Notes'])
df['Completion Date'] = pd.to_datetime(df['Completion Date'])

# Group and sort by Match ID and Completion Date
grouped = df.groupby('Match ID 18Char')

# Prepare data tuples: (sequence of notes, target final match length)
data = []
for match_id, group in grouped:
    group_sorted = group.sort_values(by='Completion Date')
    notes_sequence = group_sorted['Match Support Contact Notes'].tolist()
    final_match_length = group_sorted['Match Length'].iloc[-1]  # Target is last match length
    data.append((notes_sequence, final_match_length))

# Initialize SentenceTransformer model for text embedding
sbert = SentenceTransformer('all-MiniLM-L6-v2')

# Custom Dataset
class MatchDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        notes, target = self.data[idx]
        embeddings = sbert.encode(notes)  # Shape: (seq_len, embed_dim)
        return torch.tensor(embeddings, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)

# Collate function to handle variable sequence lengths
def collate_fn(batch):
    sequences, targets = zip(*batch)
    lengths = [seq.shape[0] for seq in sequences]
    padded_sequences = nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_sequences, torch.tensor(lengths), torch.tensor(targets)

# Model definition
class SentimentRNN(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super(SentimentRNN, self).__init__()
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        packed_input = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.rnn(packed_input)
        output = self.fc(hidden[-1])  # Use last hidden state
        return output.squeeze()

# Hyperparameters
embed_dim = 384  # Embedding size of 'all-MiniLM-L6-v2'
hidden_dim = 128
batch_size = 8

# Dataset and DataLoader
dataset = MatchDataset(data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

# Model, Loss, Optimizer
model = SentimentRNN(embed_dim, hidden_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 5
model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for padded_seqs, lengths, targets in tqdm(dataloader):
        optimizer.zero_grad()
        preds = model(padded_seqs, lengths)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss / len(dataloader):.4f}")

# Model ready for evaluation or further tuning


/Users/gautam/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
 24%|██▍       | 98/408 [01:04<02:45,  1.87it/s]